# Runtime Environment Setup

In [1]:
from pathlib import Path
import subprocess

repo_root = Path("/content/560-FaceRecognition")
if not repo_root.exists():
    subprocess.run([
        "git",
        "clone",
        "https://github.com/ethan-yountz/560-FaceRecognition",
        str(repo_root),
    ], check=True)
else:
    print(f"Using existing repo at {repo_root}")

Using existing repo at /content/560-FaceRecognition


In [2]:
import os
from pathlib import Path

project_root = Path("/content/560-FaceRecognition/project-fr")
os.chdir(project_root)
print(Path.cwd())

/content/560-FaceRecognition/project-fr


In [3]:
!pwd
!ls

/content/560-FaceRecognition/project-fr
560-FaceRecognition  make_validation_split.py  README.md
datasets	     models		       run_baseline_benchmark.py
evaluate.py	     notebooks		       train_example.py


In [4]:
from pathlib import Path
from google.colab import drive
import subprocess

drive.mount("/content/drive", force_remount=True)

drive_root = Path("/content/drive/MyDrive")
comp560_dir = drive_root / "Comp560"
candidate_paths = [
    comp560_dir / "datasets.tar.gz",
    drive_root / "datasets.tar.gz",
]
datasets_tar = next((path for path in candidate_paths if path.exists()), None)

if datasets_tar is None:
    if comp560_dir.exists():
        visible_files = sorted(path.name for path in comp560_dir.iterdir())[:20]
        raise FileNotFoundError(
            f"Could not find datasets.tar.gz. Checked: {candidate_paths}. Files in {comp560_dir}: {visible_files}"
        )
    raise FileNotFoundError(
        f"Could not find datasets.tar.gz. Checked: {candidate_paths}. Folder not found: {comp560_dir}"
    )

subprocess.run(["tar", "-xf", str(datasets_tar)], check=True)
print(f"Extracted {datasets_tar} into {Path.cwd()}")

Mounted at /content/drive
Extracted /content/drive/MyDrive/Comp560/datasets.tar.gz into /content/560-FaceRecognition/project-fr


In [5]:
from pathlib import Path
import subprocess

dataset_dir = Path("datasets/dataset_a")
if not dataset_dir.exists():
    available_datasets = sorted(path.name for path in Path("datasets").iterdir() if path.is_dir())
    raise FileNotFoundError(
        f"Missing dataset_a after extraction. Available dataset directories: {available_datasets}"
    )

images_tar = dataset_dir / "images.tar.gz"
if not images_tar.exists():
    raise FileNotFoundError(f"Missing image archive: {images_tar}")

subprocess.run(["tar", "-xf", images_tar.name], cwd=images_tar.parent, check=True)
print(f"Extracted {images_tar}")

Extracted datasets/dataset_a/images.tar.gz


In [6]:
from pathlib import Path
import time

save_target = Path("/content/drive/MyDrive/Comp560/saves")
save_target.mkdir(parents=True, exist_ok=True)
save_link = Path("saves")

if save_link.is_symlink():
    if save_link.resolve() != save_target.resolve():
        save_link.unlink()
        save_link.symlink_to(save_target, target_is_directory=True)
elif save_link.exists():
    raise FileExistsError(f"{save_link} already exists and is not a symlink.")
else:
    save_link.symlink_to(save_target, target_is_directory=True)

with open(save_link / "logs.txt", "a", encoding="utf-8") as handle:
    handle.write(time.strftime("%Y-%m-%d %H:%M:%S\n"))

print(f"Using save directory: {save_link.resolve()}")

Using save directory: /content/drive/MyDrive/Comp560/saves


In [7]:
from pathlib import Path
import subprocess

split_dir = Path("datasets/dataset_a/splits/val_15_seed42")
required_split_files = [
    split_dir / "train_metadata.parquet",
    split_dir / "val_metadata.parquet",
    split_dir / "val_pairs.parquet",
]
missing_split_files = [str(path) for path in required_split_files if not path.exists()]

if missing_split_files:
    print(f"Creating held-out split because these files are missing: {missing_split_files}")
    subprocess.run([
        "python",
        "make_validation_split.py",
        "--dataset_root",
        "datasets/dataset_a",
        "--val_fraction",
        "0.15",
        "--seed",
        "42",
    ], check=True)
else:
    print(f"Using existing held-out split: {split_dir}")

Creating held-out split because these files are missing: ['datasets/dataset_a/splits/val_15_seed42/train_metadata.parquet', 'datasets/dataset_a/splits/val_15_seed42/val_metadata.parquet', 'datasets/dataset_a/splits/val_15_seed42/val_pairs.parquet']


## Command Arguments

In [8]:
training_args = """
--data_root datasets/dataset_a
--save_dir saves/
--train_metadata splits/val_15_seed42/train_metadata.parquet
--val_metadata splits/val_15_seed42/val_metadata.parquet
--val_pairs splits/val_15_seed42/val_pairs.parquet
--backbone mobilefacenet
--embedding_dim 128
--epochs 1
""".replace("\n", " ")

prediction_args = """
--predict
--checkpoint saves/best_model.pth
--dataset_root datasets/dataset_a
--output predictions/dataset_a.csv
""".replace("\n", " ")

shared_args = """
--batch_size 256
--num_workers 8
--device cuda
""".replace("\n", " ")

evaluation_args = """
--student_id 730702719
--prediction predictions/dataset_a.csv
--acknowledge_benchmark_labels
--datasets dataset_a
""".replace("\n", " ")

sweep_args = """
--data_root datasets/dataset_a
--train_metadata splits/val_15_seed42/train_metadata.parquet
--val_metadata splits/val_15_seed42/val_metadata.parquet
--val_pairs splits/val_15_seed42/val_pairs.parquet
--save_root sweeps/mobilefacenet_first_pass
--strategy balanced
--max_trials 12
--epochs_per_trial 2
--eval_batch_size 256
--num_workers 8
--select_metric AUC
--amp
--device cuda
""".replace("\n", " ")

# MobileFaceNet Sweep
## First Pass: 12 Balanced Trials x 2 Epochs

In [9]:
cmd = f"python run_mobilefacenet_sweep.py {sweep_args}"
print(cmd)

python run_mobilefacenet_sweep.py  --data_root datasets/dataset_a --train_metadata splits/val_15_seed42/train_metadata.parquet --val_metadata splits/val_15_seed42/val_metadata.parquet --val_pairs splits/val_15_seed42/val_pairs.parquet --save_root sweeps/mobilefacenet_first_pass --strategy balanced --max_trials 12 --epochs_per_trial 2 --eval_batch_size 256 --num_workers 8 --select_metric AUC --amp --device cuda 


In [12]:
!{cmd}

Running 12 MobileFaceNet trials with 2 epochs each. Results will be saved under sweeps/mobilefacenet_first_pass.

[1/12] trial_01: {"arcface_m": 0.35, "arcface_s": 32.0, "batch_size": 128, "lr": 0.0003, "warmup_epochs": 1, "weight_decay": 1e-05}
/usr/bin/python3 train_example.py --data_root datasets/dataset_a --save_dir sweeps/mobilefacenet_first_pass/trial_01 --backbone mobilefacenet --embedding_dim 128 --loss arcface --epochs 2 --select_metric AUC --lr 0.0003 --weight_decay 1e-05 --batch_size 128 --eval_batch_size 256 --arcface_m 0.35 --arcface_s 32.0 --warmup_epochs 1 --num_workers 8 --device cuda --train_metadata splits/val_15_seed42/train_metadata.parquet --val_metadata splits/val_15_seed42/val_metadata.parquet --val_pairs splits/val_15_seed42/val_pairs.parquet --amp
Training set: 199323 images, 1571 classes using label 'component_id'
Epoch 1/2: 100% 1557/1557 [00:39<00:00, 39.25it/s, loss=7.4228, lr=0.000299] 
Epoch 1: avg_loss=14.1436, time=39.67s
  Validation AUC: 82.7472      

In [13]:
from pathlib import Path
import json
import pandas as pd

results_path = Path("sweeps/mobilefacenet_first_pass/sweep_results.csv")
if not results_path.exists():
    raise FileNotFoundError(f"Sweep results not found: {results_path}")

results = pd.read_csv(results_path)
results = results[results["status"] == "ok"].copy()
if results.empty:
    raise RuntimeError("No successful sweep trials were found.")

config_cols = ["lr", "weight_decay", "batch_size", "arcface_m", "arcface_s", "warmup_epochs"]
metric_cols = ["AUC", "TAR@FAR=1e-6", "TAR@FAR=1e-5", "TAR@FAR=1e-4", "TAR@FAR=1e-3"]
display_cols = ["trial_id"] + metric_cols + config_cols + ["save_dir"]

best_by_auc = results.sort_values("AUC", ascending=False).iloc[0]
print("Best setup by AUC")
print(best_by_auc[display_cols].to_string())

print("\nTop 5 trials by AUC")
display(results.sort_values("AUC", ascending=False)[display_cols].head(5))

tar_cols = ["TAR@FAR=1e-6", "TAR@FAR=1e-5", "TAR@FAR=1e-4", "TAR@FAR=1e-3"]
top_tar_rows = []
for metric in tar_cols:
    top_row = results.sort_values(metric, ascending=False).iloc[0]
    top_tar_rows.append({
        "metric": metric,
        "trial_id": top_row["trial_id"],
        "score": top_row[metric],
        "AUC": top_row["AUC"],
        "lr": top_row["lr"],
        "weight_decay": top_row["weight_decay"],
        "batch_size": top_row["batch_size"],
        "arcface_m": top_row["arcface_m"],
        "arcface_s": top_row["arcface_s"],
        "warmup_epochs": top_row["warmup_epochs"],
        "save_dir": top_row["save_dir"],
    })

top_tar_df = pd.DataFrame(top_tar_rows)
print("\nBest setup for each TAR metric")
display(top_tar_df)

best_config_path = Path("sweeps/mobilefacenet_first_pass/best_config.json")
if best_config_path.exists():
    print("\nSaved best_config.json")
    print(json.dumps(json.loads(best_config_path.read_text()), indent=2))

Best setup by AUC
trial_id                                         trial_08
AUC                                             86.928624
TAR@FAR=1e-6                                          0.0
TAR@FAR=1e-5                                     3.173575
TAR@FAR=1e-4                                     9.650259
TAR@FAR=1e-3                                     18.65285
lr                                                  0.001
weight_decay                                      0.00001
batch_size                                            192
arcface_m                                             0.4
arcface_s                                            64.0
warmup_epochs                                           1
save_dir         sweeps/mobilefacenet_first_pass/trial_08

Top 5 trials by AUC


,trial_id,AUC,TAR@FAR=1e-6,TAR@FAR=1e-5,TAR@FAR=1e-4,TAR@FAR=1e-3,lr,weight_decay,batch_size,arcface_m,arcface_s,warmup_epochs,save_dir
7,trial_08,86.928624,0.0,3.173575,9.650259,18.652850,0.0010,0.00001,192,0.40,64.0,1,sweeps/mobilefacenet_first_pass/trial_08
10,trial_11,86.903550,0.0,4.015544,8.549223,18.199482,0.0007,0.00001,192,0.40,32.0,1,sweeps/mobilefacenet_first_pass/trial_11
6,trial_07,86.238534,0.0,4.533679,10.297927,16.968912,0.0007,0.00030,128,0.35,48.0,2,sweeps/mobilefacenet_first_pass/trial_07
3,trial_04,86.118217,0.0,3.367876,7.189119,14.961140,0.0010,0.00030,128,0.50,32.0,2,sweeps/mobilefacenet_first_pass/trial_04
11,trial_12,85.537204,0.0,3.562176,5.051813,13.989637,0.0010,0.00003,256,0.45,48.0,2,sweeps/mobilefacenet_first_pass/trial_12



Best setup for each TAR metric


,metric,trial_id,score,AUC,lr,weight_decay,batch_size,arcface_m,arcface_s,warmup_epochs,save_dir
0,TAR@FAR=1e-6,trial_01,0.000000,85.265250,0.0003,0.00001,128,0.35,32.0,1,sweeps/mobilefacenet_first_pass/trial_01
1,TAR@FAR=1e-5,trial_07,4.533679,86.238534,0.0007,0.00030,128,0.35,48.0,2,sweeps/mobilefacenet_first_pass/trial_07
2,TAR@FAR=1e-4,trial_07,10.297927,86.238534,0.0007,0.00030,128,0.35,48.0,2,sweeps/mobilefacenet_first_pass/trial_07
3,TAR@FAR=1e-3,trial_08,18.652850,86.928624,0.0010,0.00001,192,0.40,64.0,1,sweeps/mobilefacenet_first_pass/trial_08



Saved best_config.json
{
  "trial_id": "trial_08",
  "status": "ok",
  "returncode": 0,
  "save_dir": "sweeps/mobilefacenet_first_pass/trial_08",
  "best_epoch": 2,
  "select_metric": 86.92862416066582,
  "AUC": 86.92862416066582,
  "TAR@FAR=1e-6": 0.0,
  "TAR@FAR=1e-5": 3.173575129533679,
  "TAR@FAR=1e-4": 9.650259067357512,
  "TAR@FAR=1e-3": 18.65284974093264,
  "total_time_seconds": 138.56637477874756,
  "train_images_per_second": 2876.542022813557,
  "peak_gpu_memory_reserved_mb": 5576.0,
  "lr": 0.001,
  "weight_decay": 1e-05,
  "batch_size": 192,
  "arcface_m": 0.4,
  "arcface_s": 64.0,
  "warmup_epochs": 1
}


# Trial 08 Long Run
## Retrain Selected Sweep Config For 20 Epochs And Benchmark It

In [ ]:
from pathlib import Path
import pandas as pd

selected_trial_id = "trial_08"
selected_epochs = 20

sweep_results_path = Path("sweeps/mobilefacenet_first_pass/sweep_results.csv")
if not sweep_results_path.exists():
    raise FileNotFoundError(f"Run the sweep first so this file exists: {sweep_results_path}")

sweep_results = pd.read_csv(sweep_results_path)
selected_rows = sweep_results[sweep_results["trial_id"] == selected_trial_id]
if selected_rows.empty:
    raise ValueError(f"Could not find {selected_trial_id} in {sweep_results_path}")

selected_trial = selected_rows.iloc[0]
selected_save_dir = Path(f"saves/{selected_trial_id}_epochs{selected_epochs}")
selected_checkpoint = selected_save_dir / "best_model.pth"
selected_prediction_output = Path(f"predictions/dataset_a_{selected_trial_id}_epochs{selected_epochs}.csv")
selected_benchmark_output = Path(f"results/dataset_a_{selected_trial_id}_epochs{selected_epochs}_benchmark.json")

selected_config = pd.DataFrame([{
    "trial_id": selected_trial["trial_id"],
    "lr": float(selected_trial["lr"]),
    "weight_decay": float(selected_trial["weight_decay"]),
    "batch_size": int(selected_trial["batch_size"]),
    "arcface_m": float(selected_trial["arcface_m"]),
    "arcface_s": float(selected_trial["arcface_s"]),
    "warmup_epochs": int(selected_trial["warmup_epochs"]),
    "sweep_auc": float(selected_trial["AUC"]),
    "sweep_tar_1e-6": float(selected_trial["TAR@FAR=1e-6"]),
    "sweep_tar_1e-5": float(selected_trial["TAR@FAR=1e-5"]),
    "sweep_tar_1e-4": float(selected_trial["TAR@FAR=1e-4"]),
    "sweep_tar_1e-3": float(selected_trial["TAR@FAR=1e-3"]),
}])
display(selected_config)

In [ ]:
trial08_train_cmd = " ".join([
    "python train_example.py",
    "--data_root datasets/dataset_a",
    "--train_metadata splits/val_15_seed42/train_metadata.parquet",
    "--val_metadata splits/val_15_seed42/val_metadata.parquet",
    "--val_pairs splits/val_15_seed42/val_pairs.parquet",
    f"--save_dir {selected_save_dir}",
    "--backbone mobilefacenet",
    "--embedding_dim 128",
    "--loss arcface",
    f"--epochs {selected_epochs}",
    f"--lr {float(selected_trial['lr'])}",
    f"--weight_decay {float(selected_trial['weight_decay'])}",
    f"--batch_size {int(selected_trial['batch_size'])}",
    "--eval_batch_size 256",
    f"--arcface_m {float(selected_trial['arcface_m'])}",
    f"--arcface_s {float(selected_trial['arcface_s'])}",
    f"--warmup_epochs {int(selected_trial['warmup_epochs'])}",
    "--select_metric AUC",
    "--num_workers 8",
    "--device cuda",
    "--amp",
])
print(trial08_train_cmd)

In [ ]:
!{trial08_train_cmd}

In [ ]:
trial08_benchmark_cmd = " ".join([
    "python run_baseline_benchmark.py",
    "--dataset_root datasets/dataset_a",
    f"--output {selected_prediction_output}",
    f"--metrics_output {selected_benchmark_output}",
    "--backbone mobilefacenet",
    f"--checkpoint {selected_checkpoint}",
    "--batch_size 256",
    "--num_workers 8",
    "--device cuda",
])
print(trial08_benchmark_cmd)

In [ ]:
!{trial08_benchmark_cmd}

In [ ]:
import json
import pandas as pd

if not selected_benchmark_output.exists():
    raise FileNotFoundError(f"Benchmark metrics not found: {selected_benchmark_output}")

benchmark_metrics = json.loads(selected_benchmark_output.read_text())
performance = benchmark_metrics["performance"]

def metric_value(*names):
    for name in names:
        if name in performance:
            return float(performance[name])
    raise KeyError(f"None of these metric keys were found: {names}")

accuracy_report = pd.DataFrame([
    {"Metric": "AUC", "Description": "Area under ROC curve", "Value": metric_value("AUC"), "Unit": "%"},
    {"Metric": "TAR@FAR=1e-6", "Description": "Verification rate at FAR 1e-6", "Value": metric_value("TAR@FAR=1e-6", "TAR@FAR=1e-06"), "Unit": "%"},
    {"Metric": "TAR@FAR=1e-5", "Description": "Verification rate at FAR 1e-5", "Value": metric_value("TAR@FAR=1e-5", "TAR@FAR=1e-05"), "Unit": "%"},
    {"Metric": "TAR@FAR=1e-4", "Description": "Verification rate at FAR 1e-4", "Value": metric_value("TAR@FAR=1e-4", "TAR@FAR=1e-04"), "Unit": "%"},
    {"Metric": "TAR@FAR=1e-3", "Description": "Verification rate at FAR 1e-3", "Value": metric_value("TAR@FAR=1e-3", "TAR@FAR=1e-03"), "Unit": "%"},
])

efficiency_report = pd.DataFrame([
    {"Metric": "Throughput", "Description": "Face encoding speed", "Value": float(benchmark_metrics["throughput_images_per_second"]), "Unit": "images/second"},
    {"Metric": "Peak Memory", "Description": "Maximum GPU memory usage", "Value": float(benchmark_metrics["peak_gpu_memory_mb"]), "Unit": "MB"},
    {"Metric": "Embedding Size", "Description": "Storage cost per face", "Value": int(benchmark_metrics["embedding_dim"]), "Unit": "dimensions"},
    {"Metric": "Total Time", "Description": "End-to-end evaluation time", "Value": float(benchmark_metrics["total_time_seconds"]), "Unit": "seconds"},
])

summary_report = pd.DataFrame([{
    "trial_id": selected_trial_id,
    "epochs": selected_epochs,
    "checkpoint": str(selected_checkpoint),
    "prediction_csv": str(selected_prediction_output),
    "benchmark_json": str(selected_benchmark_output),
}])

print("Selected trial retrain summary")
display(summary_report)

print("Accuracy metrics")
display(accuracy_report)

print("Efficiency metrics")
display(efficiency_report)

# Train Model

In [ ]:
cmd = f"python train_example.py {training_args} {shared_args}"
print(cmd)

In [ ]:
!{cmd}

# Evaluate Model
## Generate Predictions

In [ ]:
cmd = f"python train_example.py {prediction_args} {shared_args}"
print(cmd)

In [ ]:
!{cmd}

## Evaluate

In [ ]:
cmd = f"python evaluate.py {evaluation_args}"
print(cmd)

In [ ]:
!{cmd}